# Anime Geography: Mapping the World of Animation
---

## Objective
While anime is quintessential to Japanese culture, its production ecosystem and narrative settings often span the entire globe.
This notebook focuses on the geospatial analysis of **[Insert: Animation Studios / Anime Settings / Characters' Origins]**.

The goal is to visualize the geographic footprint of the industry and answer:
* **Global Distribution:** Where are the major hotspots located outside of Japan?
* **Density:** Which regions have the highest concentration of activity?

---

In [1]:
from lib import dbconnection as dbc
import pandas as pd
import plotly.express as px
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)

engine = dbc.create_db_engine()

## 1. Data Ingestion & Geocoding

To visualize the data on a map, we need to transform textual location data (e.g., "Tokyo", "Paris", "New York") into geographic coordinates (Latitude and Longitude).

In this step, we:
1.  **Extract locations** from the dataset.
2.  **Clean and standardize** the city/country names.
3.  **Geocode** the addresses to obtain precise coordinates for plotting.

In [2]:
query = "SELECT person_mal_id, relevant_location FROM person_details WHERE relevant_location IS NOT NULL"
df_person = pd.read_sql(query, engine)
query = "SELECT person_mal_id, anime_mal_id FROM person_anime_works"
df_works = pd.read_sql(query, engine)

In [3]:
merged_df = pd.merge(df_works, df_person, on='person_mal_id')

In [4]:
country_counts = merged_df['relevant_location'].value_counts().reset_index()
country_counts.columns = ['Country', 'Work_Count']


## 2. Interactive Map Visualization

The interactive map below displays the global distribution of the analyzed data.

**How to read this map:**
* **Markers/Pins:** Represent individual locations. Click on them to see details (e.g., Name, Count).
* **Heatmap/Clusters:** (If applicable) Show areas with high density, highlighting the "powerhouses" of the industry.

This visualization allows us to appreciate the international scale of the anime world at a glance.

In [5]:
fig = px.choropleth(
    country_counts,
    locations="Country",
    locationmode='country names',
    color="Work_Count",
    hover_data=["Country"],

    color_continuous_scale="Viridis",
    title="Global Distribution of Anime Works(Logarithmic Scale)",
    labels={'Country': 'Log10(Users)', '': 'Work_Count'}
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='equirectangular'
    ),
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

fig.write_html("../../graphs/anime_works_world_log.html")

